## Spark Aggregate Functions

### Funciones simple de agregacion

In [0]:
%run "../includes/configuration" 

In [0]:
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
display(movies_df)

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, max, min, avg

In [0]:
movies_df.select(count("*")).show()

In [0]:
movies_df.select(count("year_release_date")).show()   # tiene algunos valores nulos. por eso da menos

In [0]:
movies_df.select(countDistinct("year_release_date")).show()

In [0]:
movies_df.select(sum("budget")).display()
# display(movies_df.select(sum("budget")))

In [0]:
movies_df.filter("year_release_date = 2016") \
        .select(sum("budget"), count("movie_id")) \
        .withColumnRenamed("sum(budget)", "total_budget") \
        .withColumnRenamed("count(movie_id)", "total_movies") \
        .display()

### Group By

In [0]:
movies_df \
    .groupBy("year_release_date") \
    .sum("budget") \
    .display()

In [0]:
# mas de una agregacion
movie_group_by = movies_df \
    .groupBy("year_release_date") \
    .agg(
        sum("budget").alias("total_budget"), \
        max("budget").alias("max_budget"), \
        min("budget").alias("min_budget"), \
        avg("budget").alias("avg_budget"), \
        count("movie_id").alias("total_movies")
    )

In [0]:
display(movie_group_by)

In [0]:
# movies_df \
#     .groupBy("year_release_date") \
#     .agg(sum("budget").alias("total_budget"), count("movie_id").alias("total_movies")) \
#     .orderBy("year_release_date") \

### Window Functions

In [0]:
from pyspark.sql.functions import rank, desc, dense_rank
from pyspark.sql.window import Window

In [0]:
movies_df.select("title", "budget", "year_release_date") \
    .withColumn("rank", rank().over(Window.partitionBy("year_release_date").orderBy(desc("budget"))) ) \
    .display()

In [0]:
movies_df.select("title", "budget", "year_release_date") \
    .filter("year_release_date is not null") \
    .withColumn("rank", rank().over(Window.partitionBy("year_release_date").orderBy(desc("budget"))) ) \
    .display()

# particiona el ranking por el año y por presupuesto, ordenando de mayor a menor presupuesto. El ranking se reinicia para cada año.
# per si hay dos presupuestos iguales para el mismo ranking, se les asigna el mismo número de ranking y se salta el siguiente número. 
# Por ejemplo, si hay dos películas con el mismo presupuesto que ocupan el primer lugar, ambas recibirán un ranking de 1 y la siguiente película recibirá un ranking de 3.

In [0]:
# para salucionar  dense_rank
movie_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))
movie_dense_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))

movies_df.select("title", "budget", "year_release_date") \
    .filter("year_release_date is not null") \
    .withColumn("rank", dense_rank().over(movie_rank) ) \
    .withColumn("dense_rank", dense_rank().over(movie_dense_rank) ) \
    .display()